# 04 - Modelos Baseline (ARIMA + Random Forest)

## Objetivo
Establecer una linea base de rendimiento con modelos clasicos.
Si los modelos de Deep Learning no superan estos baselines, no justifican su complejidad.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

from src.data.preprocessor import FiberDataPreprocessor
from src.data.feature_engineer import FiberFeatureEngineer
from src.models.baseline import ARIMABaseline, RandomForestBaseline
from src.utils.visualization import apply_style, plot_confusion_matrix

apply_style()

In [ ]:
# Preparar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

fe = FiberFeatureEngineer()
train_feat = fe.create_all_features(train_df)
val_feat = fe.create_all_features(val_df)
test_feat = fe.create_all_features(test_df)

feature_names = fe.get_feature_names(train_feat)
print(f"Features: {len(feature_names)}")
print(f"Train: {train_feat.shape}, Test: {test_feat.shape}")

In [ ]:
# --- ARIMA: Prediccion univariada ---
# Seleccionar un segmento normal para demostrar ARIMA
seg_data = train_feat[train_feat['segment_id'] == 'SEG-001']['optical_power_dbm']

arima = ARIMABaseline(order=(5, 1, 2))
arima.fit(seg_data.head(500))  # Entrenar con primeras 500 lecturas

forecast = arima.predict(steps=96)  # Predecir 24 horas

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(400, 500), seg_data.iloc[400:500].values, label='Historico', color='steelblue')
ax.plot(range(500, 596), forecast, label='ARIMA Forecast', color='red', linestyle='--')
actual = seg_data.iloc[500:596].values if len(seg_data) > 596 else None
if actual is not None:
    ax.plot(range(500, 500+len(actual)), actual, label='Actual', color='green', alpha=0.7)
ax.set_xlabel('Lectura')
ax.set_ylabel('Potencia Optica (dBm)')
ax.set_title('ARIMA - Prediccion de Potencia Optica')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest: Clasificacion de fallas ---
X_train = train_feat[feature_names].values
y_train = train_feat['fault_label'].values
X_test = test_feat[feature_names].values
y_test = test_feat['fault_label'].values

rf = RandomForestBaseline()
rf.fit(X_train, y_train, feature_names=feature_names)

results = rf.evaluate(X_test, y_test)
print(f"\nAccuracy: {results['accuracy']:.4f}")
print(f"F1 Weighted: {results['f1_weighted']:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf.predict(X_test),
                            target_names=['normal', 'physical_cut', 'degradation', 'bad_splice']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, rf.predict(X_test))
fig = plot_confusion_matrix(cm, ['normal', 'physical_cut', 'degradation', 'bad_splice'],
                            title='Random Forest - Matriz de Confusion')
plt.show()

In [ ]:
# Feature Importance
importance_df = rf.get_feature_importance(top_n=20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
ax.set_xlabel('Importancia')
ax.set_title('Top 20 Features - Random Forest')
plt.tight_layout()
plt.show()

# Guardar modelo
rf.save()